In [ ]:
import zipfile

def unzip_file(zip_path, extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

unzip_file('/content/abalone.zip', '/content/')


In [ ]:
def load_data(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            row = line.strip().split(',')
            data.append(row)
    return data

data = load_data('/content/abalone.data')


In [ ]:
def encode_sex(value):
    if value == 'M':
        return 0.0
    elif value == 'F':
        return 1.0
    else:
        return 0.5

def preprocess(data):
    X = []
    Y = []

    for row in data:
        features = [encode_sex(row[0])]
        for i in range(1, 8):
            features.append(float(row[i]))
        X.append(features)
        Y.append(float(row[8]))  # Rings

    return X, Y

X, Y = preprocess(data)


In [ ]:
def min_max_normalize(X):
    mins = []
    maxs = []

    for j in range(len(X[0])):
        min_val = X[0][j]
        max_val = X[0][j]
        for i in range(len(X)):
            if X[i][j] < min_val:
                min_val = X[i][j]
            if X[i][j] > max_val:
                max_val = X[i][j]
        mins.append(min_val)
        maxs.append(max_val)

    X_norm = []
    for i in range(len(X)):
        row = []
        for j in range(len(X[0])):
            if maxs[j] - mins[j] == 0:
                row.append(0)
            else:
                row.append((X[i][j] - mins[j]) / (maxs[j] - mins[j]))
        X_norm.append(row)

    return X_norm

X = min_max_normalize(X)





In [ ]:
def train_test_split(X, Y, ratio=0.8):
    split = int(len(X) * ratio)
    return X[:split], X[split:], Y[:split], Y[split:]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y)


In [ ]:
import numpy as np
def sigmoid(x):
    return 1 / (1 + (2.71828 ** (-x)))

def sigmoid_derivative(output):
    return output * (1 - output)
def relu(x):
    if x > 0:
        return x
    else:
        return 0


In [ ]:
def relu_derivative(x):
    if x > 0:
        return 1
    else:
        return 0


In [ ]:
import random

def initialize_weights(input_size, hidden_size):
    w_ih = []
    for i in range(hidden_size):
        row = []
        for j in range(input_size):
            row.append(random.uniform(-0.5, 0.5))
        w_ih.append(row)

    w_ho = []
    for i in range(hidden_size):
        w_ho.append(random.uniform(-0.5, 0.5))

    return w_ih, w_ho



In [ ]:
def forward_pass(inputs, w_ih, w_ho):
    hidden = []

    for i in range(len(w_ih)):
        total = 0
        for j in range(len(inputs)):
            total += inputs[j] * w_ih[i][j]
        hidden.append(sigmoid(total))

    output = 0
    for i in range(len(hidden)):
        output += hidden[i] * w_ho[i]

    return hidden, output




In [ ]:
def backpropagation(inputs, hidden, output, target, w_ih, w_ho, lr):
    error = target - output
    delta_output = error  # linear output

    # hidden → output update
    for i in range(len(w_ho)):
        w_ho[i] += lr * delta_output * hidden[i]

    # input → hidden update
    for i in range(len(w_ih)):
        for j in range(len(w_ih[i])):
            delta_hidden = sigmoid_derivative(hidden[i]) * w_ho[i] * delta_output
            w_ih[i][j] += lr * delta_hidden * inputs[j]



In [ ]:
def train(X, Y, epochs, lr):
    input_size = len(X[0])
    hidden_size = 12  # ideal for sigmoid

    w_ih, w_ho = initialize_weights(input_size, hidden_size)

    for epoch in range(epochs):
        mse = 0
        for i in range(len(X)):
            hidden, output = forward_pass(X[i], w_ih, w_ho)
            mse += (Y[i] - output) ** 2
            backpropagation(X[i], hidden, output, Y[i], w_ih, w_ho, lr)

        if epoch % 20 == 0:
            print("Epoch:", epoch, "MSE:", mse / len(X))

    return w_ih, w_ho



In [ ]:
def predict(X, w_ih, w_ho):
    preds = []
    for x in X:
        _, output = forward_pass(x, w_ih, w_ho)
        preds.append(output)
    return preds


In [ ]:
weights_ih, weights_ho = train(
    X_train,
    Y_train,
    epochs=500,
    lr=0.003
)

preds = predict(X_test[:5], weights_ih, weights_ho)

print("Predicted Rings:", preds)
print("Actual Rings:", Y_test[:5])


Epoch: 0 MSE: 7.705766338340576
Epoch: 20 MSE: 3.8004704316999685
Epoch: 40 MSE: 3.554840354462604
Epoch: 60 MSE: 3.505510367861313
Epoch: 80 MSE: 3.486362072026766
Epoch: 100 MSE: 3.4786524201051394
Epoch: 120 MSE: 3.4755891673072
Epoch: 140 MSE: 3.4743305959661255
Epoch: 160 MSE: 3.473795172374013
Epoch: 180 MSE: 3.4735739684363622
Epoch: 200 MSE: 3.473499778199838
Epoch: 220 MSE: 3.473496161475811
Epoch: 240 MSE: 3.473524651758707
Epoch: 260 MSE: 3.473564598800067
Epoch: 280 MSE: 3.4736044418137406
Epoch: 300 MSE: 3.4736374935645156
Epoch: 320 MSE: 3.473659747093179
Epoch: 340 MSE: 3.473668676958134
Epoch: 360 MSE: 3.473662552642788
Epoch: 380 MSE: 3.47364001411841
Epoch: 400 MSE: 3.473599766393743
Epoch: 420 MSE: 3.4735402992222713
Epoch: 440 MSE: 3.473459559344313
Epoch: 460 MSE: 3.4733545114726216
Epoch: 480 MSE: 3.4732205345588176
Predicted Rings: [12.311332659698902, 10.237551282276137, 10.96701033801044, 12.892401124740388, 12.842426011690229]
Actual Rings: [12.0, 14.0, 13.0, 

In [ ]:
def rings_to_age(rings):
    return rings + 1.5

print("Predicted Ages:", [rings_to_age(p) for p in preds])



Predicted Ages: [13.811332659698902, 11.737551282276137, 12.46701033801044, 14.392401124740388, 14.342426011690229]


In [ ]:
def mean_absolute_error(actual, predicted):
    total = 0
    for i in range(len(actual)):
        total += abs(actual[i] - predicted[i])
    return total / len(actual)


In [ ]:
def root_mean_squared_error(actual, predicted):
    total = 0
    for i in range(len(actual)):
        total += (actual[i] - predicted[i]) ** 2
    return (total / len(actual)) ** 0.5


In [ ]:
def regression_accuracy(actual, predicted):
    mae = mean_absolute_error(actual, predicted)

    mean_actual = 0
    for v in actual:
        mean_actual += v
    mean_actual = mean_actual / len(actual)

    accuracy = 100 - (mae / mean_actual) * 100
    return accuracy


In [ ]:
test_preds = predict(X_test, weights_ih, weights_ho)

mae = mean_absolute_error(Y_test, test_preds)
rmse = root_mean_squared_error(Y_test, test_preds)
acc = regression_accuracy(Y_test, test_preds)

print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("Regression Accuracy (%):", acc)


Mean Absolute Error (MAE): 3.245285732182254
Root Mean Squared Error (RMSE): 3.550606918967846
Regression Accuracy (%): 65.86918012197302


In [ ]:
def print_comparison_table(actual, predicted, rows=10):
    print("Index\tActual Rings\tPredicted Rings")
    print("---------------------------------------")

    for i in range(rows):
        print(i, "\t", round(actual[i], 2), "\t\t", round(predicted[i], 2))

print_comparison_table(Y_test, test_preds, rows=10)


Index	Actual Rings	Predicted Rings
---------------------------------------
0 	 12.0 		 12.31
1 	 14.0 		 10.24
2 	 13.0 		 10.97
3 	 13.0 		 12.89
4 	 12.0 		 12.84
5 	 14.0 		 13.02
6 	 11.0 		 10.08
7 	 13.0 		 11.74
8 	 10.0 		 11.89
9 	 11.0 		 12.33


In [ ]:
def print_age_table(actual_rings, predicted_rings, rows=10):
    print("Index\tActual Age\tPredicted Age")
    print("-----------------------------------")

    for i in range(rows):
        actual_age = actual_rings[i] + 1.5
        predicted_age = predicted_rings[i] + 1.5
        print(i, "\t", round(actual_age, 2), "\t\t", round(predicted_age, 2))

print_age_table(Y_test, test_preds, rows=10)


Index	Actual Age	Predicted Age
-----------------------------------
0 	 13.5 		 13.81
1 	 15.5 		 11.74
2 	 14.5 		 12.47
3 	 14.5 		 14.39
4 	 13.5 		 14.34
5 	 15.5 		 14.52
6 	 12.5 		 11.58
7 	 14.5 		 13.24
8 	 11.5 		 13.39
9 	 12.5 		 13.83
